# 03 · Várias cabeças de atenção

**Peça nova:** `MultiHeadAttention`. Uma cabeça só consegue fazer **um tipo de pergunta** por posição (uma query). Com várias cabeças em paralelo, cada uma pode procurar uma coisa diferente: a vogal anterior, o início da palavra, o último espaço…

O custo não aumenta: com `n_head` cabeças, cada uma tem `d_model / n_head` dimensões. As saídas são concatenadas (voltando a `d_model`) e passam por uma **projeção linear** que mistura o que as cabeças encontraram.

Daqui em diante, `Head` vem pronta de `lab.model` (é a mesma do notebook 02). Diário: `F1-MHA`.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from lab.data import shakespeare
from lab.model import Head, LanguageModel, count_params
from lab.plot import compare, plot_attention, table
from lab.train import estimate_loss, pick_device, train

DEVICE = pick_device()
ds = shakespeare()
tok, V = ds.tok, ds.tok.vocab_size
BLOCK, D, STEPS, LR = 32, 64, 3000, 1e-3
print(ds, "| dispositivo:", DEVICE)

## A peça: `MultiHeadAttention`

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head, block_size):
        super().__init__()
        assert d_model % n_head == 0
        head_size = d_model // n_head
        self.heads = nn.ModuleList([Head(d_model, head_size, block_size) for _ in range(n_head)])
        self.proj = nn.Linear(n_head * head_size, d_model)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)  # (B, T, n_head * head_size) = (B, T, d_model)
        return self.proj(out)

### Teste: formato e número de parâmetros
Com `d_model` fixo, o número de parâmetros da atenção **não depende** de quantas cabeças você usa.

In [ ]:
x = torch.randn(2, BLOCK, D)
for nh in [1, 4, 16]:
    mha = MultiHeadAttention(D, nh, BLOCK)
    print(f"{nh:2d} cabeças de {D // nh:2d} dimensões: saída {tuple(mha(x).shape)}, {count_params(mha):,} parâmetros")

## O modelo

In [ ]:
class MultiHeadLM(LanguageModel):
    def __init__(self, vocab_size, block_size=32, d_model=64, n_head=4):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.sa = MultiHeadAttention(d_model, n_head, block_size)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def logits(self, idx):
        T = idx.size(1)
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        x = self.sa(x)
        return self.lm_head(x)

## Treino de referência (4 cabeças)

In [ ]:
torch.manual_seed(1337)
model = MultiHeadLM(V, BLOCK, D, n_head=4)
run = train(model, ds, name="03-cabecas", steps=STEPS, lr=LR, device=DEVICE)
print(run.sample)

In [ ]:
compare(["01-bigram", "02-atencao", "03-cabecas"], title="Bigrama → 1 cabeça → 4 cabeças")
table(["01-bigram", "02-atencao", "03-cabecas"])

### O que cada cabeça olha

In [ ]:
text = "First Citizen:\nBefore we proceed any further"[:BLOCK]
idx = torch.tensor([tok.encode(text)], device=DEVICE)
model.eval()
model(idx)
fig, axs = plt.subplots(1, len(model.sa.heads), figsize=(5 * len(model.sa.heads), 5))
for i, (ax, h) in enumerate(zip(axs, model.sa.heads)):
    plot_attention(h.last_att[0].cpu(), list(text), ax=ax, title=f"cabeça {i}")
plt.tight_layout()
plt.show()

---
## Experimentos

### 1. Quantas cabeças?
Mesmo `d_model = 64`, de 1 cabeça de 64 dimensões até 16 cabeças de 4.

**Minha previsão:** _(existe um ponto ótimo? o que acontece com cabeças de 4 dimensões? o tempo por passo muda?)_

In [ ]:
for nh in [1, 2, 4, 8, 16]:
    torch.manual_seed(1337)
    train(MultiHeadLM(V, BLOCK, D, nh), ds, name=f"03-nh{nh:02d}", steps=STEPS, lr=LR, device=DEVICE, verbose=False, sample_tokens=0)
compare(prefix="03-nh", title="Número de cabeças (d_model = 64)")
table(prefix="03-nh")

### 2. Desligar uma cabeça de cada vez (ablação)
Num modelo treinado, zere a saída de uma cabeça e meça quanto a loss de validação piora. É a forma mais simples de perguntar "o que essa cabeça faz?".

**Minha previsão:** _(todas importam igual, ou uma ou duas carregam o modelo?)_

In [ ]:
def val_loss(m):
    torch.manual_seed(0)  # mesmos batches para todas as medições
    return estimate_loss(m, ds, batch_size=64, block_size=BLOCK, eval_batches=40, device=DEVICE)["val"]

base = val_loss(model)
print(f"sem ablação: {base:.4f}")
for i, h in enumerate(model.sa.heads):
    hook = h.register_forward_hook(lambda mod, inp, out: torch.zeros_like(out))
    loss = val_loss(model)
    print(f"sem a cabeça {i}: {loss:.4f}  (+{loss - base:.4f})")
    hook.remove()

---
## Para o diário (`F1-MHA-01`)

- Qual número de cabeças funcionou melhor com `d_model = 64`? Por que cabeças muito estreitas atrapalham?
- As cabeças aprenderam padrões diferentes? Qual delas a ablação mostrou ser mais importante, e o que ela olha no mapa?

**Próximo:** [04 · MLP](04_mlp.ipynb).